# P7v2: CGAR — Confidence-Guided Adaptive Retrieval
## Two-Level Controller: OptStop (Candidate Budget) + AdaGraph (Path Relevance)

**Key insight:** P7's path selection had only 6 paths to choose from → no room for improvement.
CGAR solves this by operating at TWO levels:
- **Level 1 (OptStop):** Vary K candidates (5→10→20→50), stop when confident
- **Level 2 (AdaGraph):** Rank paths by relevance (not diversity), penalize redundancy (MMR)

**Why this works:** Decision space is 10× larger (K varies), SCST has real variance.

In [ ]:
import os, sys, time, json, random, pickle, string, re
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import defaultdict

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

# Paths
BASE_DIR = '/content/drive/MyDrive/ThS/Research/colab/graph_rag/'
CHECKPOINT_DIR = os.path.join(BASE_DIR, 'ckpt')
RESULTS_DIR = os.path.join(BASE_DIR, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Mounted at /content/drive
Device: cuda
GPU: NVIDIA A100-SXM4-80GB
Memory: 85.1 GB


In [ ]:
# Add project to path

GRL_RAG_DIR = BASE_DIR
sys.path.insert(0, GRL_RAG_DIR)
from grl_rag import (GNNHeuristic, Embedder, extract_candidate_paths,
                      path_to_string, compute_reward, GreedyGNNRetriever,
                      PathSelectionPolicy, PathSelectionRetriever)

# Load GNN features (Phase 2 output)
print("Loading GNN features...")
gnn_train = GNNHeuristic(os.path.join(CHECKPOINT_DIR, 'gnn_features_train.pkl'))
gnn_test = GNNHeuristic(os.path.join(CHECKPOINT_DIR, 'gnn_features_test.pkl'))

# SBERT embedder with cache
EMB_CACHE = os.path.join(CHECKPOINT_DIR, 'embeddings_cache.pkl')
embedder = Embedder(cache_path=EMB_CACHE)
EMB_DIM = embedder.emb_dim
print(f"EMB_DIM: {EMB_DIM}")

train_qids = list(gnn_train.features.keys())
test_qids = list(gnn_test.features.keys())
print(f"Train: {len(train_qids)} questions, Test: {len(test_qids)} questions")

# Quick stats
sample_qid = test_qids[0]
sample_paths = extract_candidate_paths(sample_qid, gnn_test, max_candidates=10)
print(f"\nSample question: {gnn_test.get(sample_qid)['question']}")
print(f"Seeds: {gnn_test.get_seeds(sample_qid)}")
print(f"Answers: {gnn_test.get_answers(sample_qid)}")
print(f"Candidate paths: {len(sample_paths)}")
for p in sample_paths[:3]:
    print(f"  {p['path_text']} (score={p['gnn_score']:.3f}, len={p['length']})")

Loading GNN features...
Loaded GNN features for 2848 questions
Loaded GNN features for 1639 questions


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

... (6 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded 22355 cached embeddings
EMB_DIM: 384
Train: 2848 questions, Test: 1639 questions

Sample question: what does jamaican people speak
Seeds: ['Jamaica']
Answers: ['Jamaican English', 'Jamaican Creole English Language']
Candidate paths: 2
  Jamaica -> language.human_language.countries_spoken_in -> Jamaican English (score=0.519, len=1)
  Jamaica -> location.country.languages_spoken -> Jamaican Creole English Language (score=0.481, len=1)


In [ ]:
"""
CGAR: Confidence-Guided Adaptive Retrieval
=============================================
Fuses OptStop (confidence-based stopping on candidate expansion)
with AdaGraph (relevance-based path ranking) in a two-level controller
on top of GNN-RAG.

Level 1 — Candidate Budget (OptStop):
  Iteratively expand K (5→10→20→50 candidates).
  At each K, estimate confidence that answer is reachable.
  SPRT: stop expanding when confidence > β.

Level 2 — Path Relevance Ranking (AdaGraph):
  Score each path by query-relevance (not marginal utility).
  Penalize redundancy via pairwise cosine similarity.
  Select top-M paths under budget.

Why this works when P7 didn't:
  P7 selected 5 from 6 paths → no room for improvement.
  CGAR varies K (5→50) → decision space 10× larger.
  CGAR ranks by relevance → LLM-aligned (not diversity-first).
"""




# ══════════════════════════════════════════════════════════════
# 1. Confidence Head — OptStop component
# ══════════════════════════════════════════════════════════════

class CandidateConfidenceHead(nn.Module):
    """Estimates P(answer reachable | top-K candidates, query).

    Input: [evidence_state, query_emb, K_features]
    Output: confidence ∈ (0, 1)

    When confidence > β → stop expanding candidates.
    """
    def __init__(self, emb_dim=384, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_dim * 2 + 4, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid(),
        )

    def forward(self, evidence, query_emb, k_features):
        """
        Args:
            evidence: (B, emb_dim) — accumulated evidence
            query_emb: (B, emb_dim)
            k_features: (B, 4) — [K/K_max, n_paths/max_paths,
                                   mean_gnn_score, coverage_estimate]
        """
        x = torch.cat([evidence, query_emb, k_features], dim=-1)
        return self.net(x).squeeze(-1)


# ══════════════════════════════════════════════════════════════
# 2. Path Relevance Head — AdaGraph component
# ══════════════════════════════════════════════════════════════

class PathRelevanceHead(nn.Module):
    """Scores each path by query-relevance (LLM-aligned).

    Unlike MarginalUtilityHead which optimizes diversity,
    this optimizes RELEVANCE to question → better LLM answers.
    """
    def __init__(self, emb_dim=384, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_dim * 2 + 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, path_embs, query_emb, features):
        """
        Args:
            path_embs: (B, N, emb_dim) — path embeddings
            query_emb: (B, emb_dim)
            features: (B, N, 2) — [gnn_score, path_length_norm]
        Returns:
            scores: (B, N) — relevance scores (raw logits)
        """
        n = path_embs.size(1)
        q_exp = query_emb.unsqueeze(1).expand(-1, n, -1)
        x = torch.cat([path_embs, q_exp, features], dim=-1)
        return self.net(x).squeeze(-1)


# ══════════════════════════════════════════════════════════════
# 3. Evidence Accumulator (GRU)
# ══════════════════════════════════════════════════════════════

class EvidenceGRU(nn.Module):
    """GRU accumulates evidence across candidate expansion rounds."""
    def __init__(self, emb_dim=384):
        super().__init__()
        self.gru = nn.GRUCell(emb_dim, emb_dim)
        self.init_proj = nn.Linear(emb_dim, emb_dim)

    def initialize(self, query_emb):
        return self.init_proj(query_emb)

    def update(self, state, new_evidence):
        return self.gru(new_evidence, state)


# ══════════════════════════════════════════════════════════════
# 4. CGAR Controller — Two-Level Unified
# ══════════════════════════════════════════════════════════════

class CGARController(nn.Module):
    """Confidence-Guided Adaptive Retrieval Controller.

    Two levels:
      Level 1 (OptStop): Decide how many candidates K to explore.
      Level 2 (AdaGraph): Rank and select paths by relevance.

    Training:
      Stage 1: Supervised (confidence calibration + relevance ranking)
      Stage 2: SCST (vary K stochastically → real advantage signal)
    """

    # Candidate budget options — provides real variance for SCST
    K_OPTIONS = [5, 10, 20, 50]

    def __init__(self, emb_dim=384, hidden_dim=256,
                 max_paths=10, sprt_beta=0.85,
                 redundancy_lambda=0.3):
        super().__init__()
        self.emb_dim = emb_dim
        self.max_paths = max_paths
        self.sprt_beta = sprt_beta
        self.redundancy_lambda = redundancy_lambda

        # Level 1: OptStop
        self.confidence_head = CandidateConfidenceHead(emb_dim, hidden_dim // 2)
        self.evidence_gru = EvidenceGRU(emb_dim)

        # Level 2: AdaGraph
        self.relevance_head = PathRelevanceHead(emb_dim, hidden_dim)

        # K-selection policy (for SCST stochastic sampling)
        self.k_policy = nn.Sequential(
            nn.Linear(emb_dim * 2 + 4, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, len(self.K_OPTIONS)),
        )

    def forward(self, query_emb, all_paths_by_k, embedder,
                deterministic=True, return_details=False):
        """
        Args:
            query_emb: (1, emb_dim) — query embedding
            all_paths_by_k: dict {K: [path_dicts]} — paths for each K option
            embedder: Embedder instance for encoding paths
            deterministic: greedy (True) or stochastic (False)

        Returns:
            selected_paths: list of selected path dicts
            details: dict with per-level info
        """
        dev = query_emb.device
        evidence = self.evidence_gru.initialize(query_emb)

        # ═══════════════════════════════════
        # LEVEL 1: Candidate expansion (OptStop)
        # ═══════════════════════════════════
        level1_details = []
        chosen_k = self.K_OPTIONS[-1]  # default: max K
        chosen_paths = []
        k_confidences = []

        for ki, K in enumerate(self.K_OPTIONS):
            paths_k = all_paths_by_k.get(K, [])
            if not paths_k:
                k_confidences.append(torch.tensor([0.0], device=dev))
                continue

            # Encode paths at this K level
            path_embs_list = []
            gnn_scores_list = []
            for p in paths_k:
                texts = []
                for h, r, t in p['path_triples']:
                    texts.extend([str(h), str(r), str(t)])
                texts = list(set(texts))
                embs = embedder.encode_batch(texts)
                path_embs_list.append(embs.mean(dim=0).clone())
                gnn_scores_list.append(p['gnn_score'])

            if not path_embs_list:
                k_confidences.append(torch.tensor([0.0], device=dev))
                continue

            path_embs_k = torch.stack(path_embs_list).to(dev)  # (N_k, D)
            mean_path_emb = path_embs_k.mean(0, keepdim=True)  # (1, D)

            # Update evidence with paths at this K
            evidence = self.evidence_gru.update(evidence, mean_path_emb)

            # K-level features
            mean_gnn = np.mean(gnn_scores_list) if gnn_scores_list else 0.0
            k_features = torch.tensor([[
                K / max(self.K_OPTIONS),
                len(paths_k) / self.max_paths,
                mean_gnn,
                ki / len(self.K_OPTIONS),
            ]], device=dev, dtype=torch.float32)

            # Confidence: is answer reachable with top-K candidates?
            conf = self.confidence_head(evidence, query_emb, k_features)
            k_confidences.append(conf)

            level1_details.append({
                'K': K, 'n_paths': len(paths_k),
                'confidence': conf, 'mean_gnn': mean_gnn,
            })

            # SPRT stopping check
            if deterministic and conf.item() > self.sprt_beta:
                chosen_k = K
                chosen_paths = paths_k
                break

            chosen_k = K
            chosen_paths = paths_k

        # Stochastic K selection (for SCST)
        if not deterministic and k_confidences:
            conf_stack = torch.stack([c.squeeze() for c in k_confidences
                                      if c.numel() > 0])
            k_logits = self.k_policy(torch.cat([
                evidence, query_emb,
                torch.tensor([[
                    len(chosen_paths) / self.max_paths,
                    conf_stack.max().item(),
                    conf_stack.mean().item(),
                    len(conf_stack) / len(self.K_OPTIONS),
                ]], device=dev, dtype=torch.float32)
            ], dim=-1))

            valid_k_count = len([c for c in k_confidences if c.numel() > 0 and c.item() > 0])
            if valid_k_count > 0:
                k_dist = torch.distributions.Categorical(
                    logits=k_logits[:, :len(self.K_OPTIONS)])
                k_idx = k_dist.sample().item()
                k_idx = min(k_idx, len(self.K_OPTIONS) - 1)
                chosen_k = self.K_OPTIONS[k_idx]
                chosen_paths = all_paths_by_k.get(chosen_k, chosen_paths)

        if not chosen_paths:
            if return_details:
                return [], {'level1': level1_details, 'level2': [],
                            'chosen_k': 0, 'n_selected': 0,
                            'final_confidence': torch.tensor([0.0], device=dev)}
            return []

        # ═══════════════════════════════════
        # LEVEL 2: Path relevance ranking (AdaGraph)
        # ═══════════════════════════════════

        # Encode chosen paths
        path_embs_list = []
        gnn_scores_list = []
        lengths_list = []
        for p in chosen_paths:
            texts = []
            for h, r, t in p['path_triples']:
                texts.extend([str(h), str(r), str(t)])
            texts = list(set(texts))
            embs = embedder.encode_batch(texts)
            path_embs_list.append(embs.mean(dim=0).clone())
            gnn_scores_list.append(p['gnn_score'])
            lengths_list.append(float(p['length']))

        path_embs = torch.stack(path_embs_list).unsqueeze(0).to(dev)  # (1, N, D)
        features = torch.stack([
            torch.tensor(gnn_scores_list, device=dev),
            torch.tensor(lengths_list, device=dev) / 5.0,
        ], dim=-1).unsqueeze(0)  # (1, N, 2)

        # Relevance scores
        rel_scores = self.relevance_head(path_embs, query_emb, features)  # (1, N)

        # Pairwise redundancy penalty
        path_norm = F.normalize(path_embs.squeeze(0), dim=-1)  # (N, D)
        sim_matrix = torch.mm(path_norm, path_norm.t())  # (N, N)

        # Greedy selection with redundancy penalty (MMR-style)
        n_paths = len(chosen_paths)
        n_select = min(self.max_paths, n_paths)

        selected_indices = []
        remaining = set(range(n_paths))
        adjusted_scores = rel_scores.squeeze(0).clone()

        for _ in range(n_select):
            if not remaining:
                break

            # Mask already selected
            mask = torch.full((n_paths,), float('-inf'), device=dev)
            for idx in remaining:
                mask[idx] = adjusted_scores[idx]

            best_idx = mask.argmax().item()
            if mask[best_idx] == float('-inf'):
                break

            selected_indices.append(best_idx)
            remaining.discard(best_idx)

            # Penalize similar paths (MMR)
            for idx in list(remaining):
                penalty = self.redundancy_lambda * sim_matrix[best_idx, idx]
                adjusted_scores[idx] = adjusted_scores[idx] - penalty

        selected_paths = [chosen_paths[i] for i in selected_indices]

        # Final confidence
        final_conf_features = torch.tensor([[
            chosen_k / max(self.K_OPTIONS),
            len(selected_paths) / self.max_paths,
            np.mean([p['gnn_score'] for p in selected_paths]) if selected_paths else 0,
            len(selected_paths) / max(len(chosen_paths), 1),
        ]], device=dev, dtype=torch.float32)
        final_conf = self.confidence_head(evidence, query_emb, final_conf_features)

        if return_details:
            return selected_paths, {
                'level1': level1_details,
                'chosen_k': chosen_k,
                'n_candidates': len(chosen_paths),
                'n_selected': len(selected_paths),
                'relevance_scores': rel_scores.detach(),
                'final_confidence': final_conf,
                'k_confidences': k_confidences,
            }
        return selected_paths


# ══════════════════════════════════════════════════════════════
# 5. CGAR Retriever
# ══════════════════════════════════════════════════════════════

class CGARRetriever:
    """End-to-end retriever: GNN-RAG heuristic → CGAR Controller → paths."""

    def __init__(self, gnn, controller, embedder,
                 max_paths_per_pair=3, device='cpu'):
        self.gnn = gnn
        self.controller = controller.to(device)
        self.embedder = embedder
        self.max_paths_per_pair = max_paths_per_pair
        self.device = device

    def retrieve(self, qid, deterministic=True, return_details=False):
        qdata = self.gnn.get(qid)
        if qdata is None:
            return {'triples': [], 'cost': 0, 'context': '',
                    'selected_paths': [], 'all_paths': {}}

        # Pre-compute paths for ALL K options
        all_paths_by_k = {}
        for K in self.controller.K_OPTIONS:
            paths = extract_candidate_paths(
                qid, self.gnn,
                max_candidates=K,
                max_paths_per_pair=self.max_paths_per_pair)
            all_paths_by_k[K] = paths

        query_emb = self.embedder.encode(
            qdata['question']).clone().unsqueeze(0).to(self.device)

        if return_details:
            selected, details = self.controller(
                query_emb, all_paths_by_k, self.embedder,
                deterministic=deterministic, return_details=True)
        else:
            selected = self.controller(
                query_emb, all_paths_by_k, self.embedder,
                deterministic=deterministic)
            details = None

        all_triples = []
        for p in selected:
            all_triples.extend(p['path_triples'])

        result = {
            'triples': all_triples,
            'cost': len(selected),
            'context': '\n'.join(p['path_text'] for p in selected),
            'selected_paths': selected,
            'all_paths': all_paths_by_k,
            'n_total_paths': sum(len(v) for v in all_paths_by_k.values()),
            'n_selected': len(selected),
        }
        if details:
            result['details'] = details
        return result


# ══════════════════════════════════════════════════════════════
# 6. Loss Function
# ══════════════════════════════════════════════════════════════

class CGARLoss(nn.Module):
    """Loss for CGAR Controller.

    L = w_conf * L_confidence + w_rel * L_relevance

    L_confidence: Calibrate candidate confidence to actual recall at each K
    L_relevance:  BCE — gold-entity paths should score higher
    """
    def __init__(self, w_conf=0.5, w_rel=0.5):
        super().__init__()
        self.w_conf = w_conf
        self.w_rel = w_rel

    def forward(self, details, gold_entities, all_paths_by_k):
        device = details['final_confidence'].device

        # ── L_confidence: calibrate at each K ──
        conf_loss = torch.tensor(0.0, device=device)
        n_conf = 0
        for ld in details['level1']:
            K = ld['K']
            paths_k = all_paths_by_k.get(K, [])
            # Actual: what fraction of gold entities reachable at this K?
            reachable = set()
            for p in paths_k:
                for h, r, t in p['path_triples']:
                    reachable.add(str(h)); reachable.add(str(t))
            actual = len(reachable & gold_entities) / max(len(gold_entities), 1)
            target = torch.tensor([actual], device=device)
            conf_loss = conf_loss + F.mse_loss(ld['confidence'], target)
            n_conf += 1
        conf_loss = conf_loss / max(n_conf, 1)

        # ── L_confidence final ──
        # What fraction of gold in selected paths?
        # (passed via details from training loop)
        final_conf = details['final_confidence']
        # Placeholder — actual recall computed externally
        conf_loss = conf_loss + F.mse_loss(final_conf,
                                            torch.tensor([0.5], device=device)) * 0.1

        # ── L_relevance: paths with gold entities should score high ──
        rel_loss = torch.tensor(0.0, device=device)
        if 'relevance_scores' in details and details['relevance_scores'] is not None:
            chosen_k = details['chosen_k']
            paths = all_paths_by_k.get(chosen_k, [])
            if paths:
                targets = torch.zeros(1, len(paths), device=device)
                for i, p in enumerate(paths):
                    ents = set()
                    for h, r, t in p['path_triples']:
                        ents.add(str(h)); ents.add(str(t))
                    if ents & gold_entities:
                        targets[0, i] = 1.0
                # BCE on relevance scores
                rel_scores = details['relevance_scores']
                if rel_scores.size(-1) == targets.size(-1):
                    rel_loss = F.binary_cross_entropy_with_logits(
                        rel_scores, targets, reduction='mean')

        total = self.w_conf * conf_loss + self.w_rel * rel_loss
        return {
            'total': total,
            'confidence': conf_loss.item(),
            'relevance': rel_loss.item() if isinstance(rel_loss, torch.Tensor) else rel_loss,
        }


# ══════════════════════════════════════════════════════════════
# 7. SCST Trainer
# ══════════════════════════════════════════════════════════════

class CGARSCSTTrainer:
    """SCST for CGAR.

    Key advantage over P7: K varies stochastically among {5,10,20,50}
    → greedy and sampled produce DIFFERENT paths → real advantage signal.
    """
    def __init__(self, controller, gnn, embedder,
                 alpha=0.6, beta=0.2, gamma=0.2,
                 n_samples=8, lr=1e-5, device='cpu'):
        self.controller = controller
        self.gnn = gnn
        self.embedder = embedder
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.n_samples = n_samples
        self.device = device
        self.optimizer = torch.optim.AdamW(
            controller.parameters(), lr=lr, weight_decay=0.01)

    def compute_reward(self, selected_paths, gold_entities, max_paths):
        if not selected_paths:
            return -0.1
        collected = set()
        for p in selected_paths:
            for h, r, t in p['path_triples']:
                collected.add(str(h)); collected.add(str(t))
        overlap = len(collected & gold_entities)
        coverage = min(overlap / max(len(gold_entities), 1), 1.0)
        efficiency = 1.0 - len(selected_paths) / max_paths
        return self.alpha * coverage + self.beta * efficiency + self.gamma * 0.5

    def train_step(self, qid):
        qdata = self.gnn.get(qid)
        if qdata is None:
            return None

        gold_entities = set(str(a) for a in self.gnn.get_answers(qid))
        if not gold_entities:
            return None

        # Pre-compute all K paths
        all_paths_by_k = {}
        for K in self.controller.K_OPTIONS:
            all_paths_by_k[K] = extract_candidate_paths(
                qid, self.gnn, max_candidates=K, max_paths_per_pair=3)

        if not any(all_paths_by_k.values()):
            return None

        query_emb = self.embedder.encode(
            qdata['question']).clone().unsqueeze(0).to(self.device)

        # Greedy baseline
        self.controller.eval()
        with torch.no_grad():
            greedy_paths = self.controller(
                query_emb, all_paths_by_k, self.embedder, deterministic=True)
        baseline_r = self.compute_reward(
            greedy_paths, gold_entities, self.controller.max_paths)

        # Stochastic samples — K varies → different paths → real variance
        self.controller.train()
        loss_terms = []
        sample_rewards = []

        for _ in range(self.n_samples):
            sel_paths, details = self.controller(
                query_emb, all_paths_by_k, self.embedder,
                deterministic=False, return_details=True)
            reward = self.compute_reward(
                sel_paths, gold_entities, self.controller.max_paths)
            sample_rewards.append(reward)

            # Actual recall for confidence target
            if sel_paths:
                collected = set()
                for p in sel_paths:
                    for h, r, t in p['path_triples']:
                        collected.add(str(h)); collected.add(str(t))
                actual_recall = len(collected & gold_entities) / max(len(gold_entities), 1)
            else:
                actual_recall = 0.0

            # Loss: advantage-weighted confidence calibration
            target = torch.tensor([actual_recall], device=self.device)
            conf_loss = F.mse_loss(details['final_confidence'], target)
            advantage = reward - baseline_r
            loss_terms.append(advantage * conf_loss + 0.01 * conf_loss)

        scst_loss = sum(loss_terms) / len(loss_terms)
        avg_reward = sum(sample_rewards) / len(sample_rewards)

        self.optimizer.zero_grad()
        scst_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.controller.parameters(), 0.5)
        self.optimizer.step()

        return {
            'avg_reward': avg_reward,
            'baseline': baseline_r,
            'loss': scst_loss.item(),
        }


# ══════════════════════════════════════════════════════════════
# 8. Evaluation
# ══════════════════════════════════════════════════════════════

def evaluate_cgar(retriever, gnn, qids, verbose=False):
    """Evaluate CGAR retriever."""
    results = defaultdict(list)
    for qid in qids:
        gold = set(str(a) for a in gnn.get_answers(qid))
        if not gold:
            continue
        ret = retriever.retrieve(qid, deterministic=True)
        collected = set()
        for h, r, t in ret['triples']:
            collected.add(str(h)); collected.add(str(t))
        overlap = len(collected & gold)
        cov = overlap / max(len(gold), 1)
        prec = overlap / max(len(collected), 1)
        f1 = 2*prec*cov / max(prec+cov, 1e-8)
        results['coverage'].append(cov)
        results['precision'].append(prec)
        results['recall'].append(cov)
        results['f1'].append(f1)
        results['cost'].append(ret['cost'])
        results['chosen_k'].append(ret.get('details', {}).get('chosen_k', 0)
                                    if ret.get('details') else 0)
        if verbose and len(results['f1']) <= 5:
            det = ret.get('details', {})
            print(f"  Q={qid}: K={det.get('chosen_k','?')} "
                  f"sel={ret['n_selected']}/{ret.get('n_total_paths','?')} "
                  f"cov={cov:.3f} prec={prec:.3f} f1={f1:.3f}")
    return {k: np.mean(v) if v else 0 for k, v in results.items()}


print("[OK] CGAR Controller loaded")
print("  Level 1: CandidateConfidenceHead (OptStop)")
print("  Level 2: PathRelevanceHead (AdaGraph)")
print("  CGARController, CGARRetriever, CGARLoss, CGARSCSTTrainer")

[OK] CGAR Controller loaded
  Level 1: CandidateConfidenceHead (OptStop)
  Level 2: PathRelevanceHead (AdaGraph)
  CGARController, CGARRetriever, CGARLoss, CGARSCSTTrainer


In [ ]:
# CGAR Configuration
HIDDEN_DIM    = 256
MAX_PATHS     = 10         # budget per question
SPRT_BETA     = 0.85       # confidence threshold (lower = fewer candidates = faster)
REDUNDANCY_LAMBDA = 0.3    # MMR redundancy penalty

# Training
S1_EPOCHS     = 10
S1_LR         = 3e-4
S2_STEPS      = 300        # MORE SCST steps (real variance now)
S2_LR         = 5e-5       # slightly higher for faster learning
S2_SAMPLES    = 8

# Reward
ALPHA_REWARD  = 0.6
BETA_REWARD   = 0.2
GAMMA_REWARD  = 0.2

MAX_PATHS_PER_PAIR = 3

print("=" * 60)
print("CGAR Configuration")
print("=" * 60)
print(f"  Hidden: {HIDDEN_DIM}, Max paths: {MAX_PATHS}")
print(f"  SPRT \u03b2: {SPRT_BETA}, Redundancy \u03bb: {REDUNDANCY_LAMBDA}")
print(f"  K options: {CGARController.K_OPTIONS}")
print(f"  S1: {S1_EPOCHS} epochs, lr={S1_LR}")
print(f"  S2: {S2_STEPS} steps, lr={S2_LR}, samples={S2_SAMPLES}")

CGAR Configuration
  Hidden: 256, Max paths: 10
  SPRT β: 0.85, Redundancy λ: 0.3
  K options: [5, 10, 20, 50]
  S1: 10 epochs, lr=0.0003
  S2: 300 steps, lr=5e-05, samples=8


In [ ]:
print("=" * 60)
print("Building CGAR Controller")
print("=" * 60)

controller = CGARController(
    emb_dim=EMB_DIM, hidden_dim=HIDDEN_DIM,
    max_paths=MAX_PATHS, sprt_beta=SPRT_BETA,
    redundancy_lambda=REDUNDANCY_LAMBDA,
).to(device)

cp = sum(p.numel() for p in controller.parameters() if p.requires_grad)
print(f"  Parameters: {cp:,} = {cp/1e6:.2f}M")
print(f"  Level 1 (OptStop):")
print(f"    ConfidenceHead: {sum(p.numel() for p in controller.confidence_head.parameters()):,}")
print(f"    EvidenceGRU:    {sum(p.numel() for p in controller.evidence_gru.parameters()):,}")
print(f"    K-Policy:       {sum(p.numel() for p in controller.k_policy.parameters()):,}")
print(f"  Level 2 (AdaGraph):")
print(f"    RelevanceHead:  {sum(p.numel() for p in controller.relevance_head.parameters()):,}")

retriever = CGARRetriever(
    gnn_test, controller, embedder,
    max_paths_per_pair=MAX_PATHS_PER_PAIR,
    device=device,
)

# Quick test
print(f"\n  Quick test on {test_qids[0]}...")
with torch.no_grad():
    test_result = retriever.retrieve(test_qids[0], deterministic=True, return_details=True)
print(f"  Chosen K: {test_result['details']['chosen_k']}")
print(f"  Selected: {test_result['n_selected']}/{test_result.get('n_total_paths','?')} paths")
print(f"  Final conf: {test_result['details']['final_confidence'].item():.3f}")

Building CGAR Controller
  Parameters: 1,472,774 = 1.47M
  Level 1 (OptStop):
    ConfidenceHead: 107,521
    EvidenceGRU:    1,034,880
    K-Policy:       99,460
  Level 2 (AdaGraph):
    RelevanceHead:  230,913

  Quick test on WebQTest-0...
  Chosen K: 50
  Selected: 2/8 paths
  Final conf: 0.490


---
## Stage 1: Supervised Pretraining
Two objectives:
1. **Confidence calibration (Level 1):** Predict actual recall at each K
2. **Path relevance ranking (Level 2):** Gold-entity paths score higher

In [ ]:
print("=" * 60)
print("STAGE 1: SUPERVISED PRETRAINING")
print("=" * 60)

loss_fn = CGARLoss(w_conf=0.5, w_rel=0.5)
optimizer = torch.optim.AdamW(controller.parameters(), lr=S1_LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=S1_EPOCHS)

s1_history = []
best_eval_f1 = 0.0

for epoch in range(S1_EPOCHS):
    controller.train()
    ep_loss, ep_conf, ep_rel, n_batch = 0, 0, 0, 0
    t0 = time.time()
    random.shuffle(train_qids)

    for qi, qid in enumerate(train_qids):
        qdata = gnn_train.get(qid)
        if qdata is None: continue

        gold_entities = set(str(a) for a in gnn_train.get_answers(qid))
        if not gold_entities: continue

        # Pre-compute paths for all K
        all_paths_by_k = {}
        for K in CGARController.K_OPTIONS:
            all_paths_by_k[K] = extract_candidate_paths(
                qid, gnn_train, max_candidates=K, max_paths_per_pair=MAX_PATHS_PER_PAIR)
        if not any(all_paths_by_k.values()): continue

        query_emb = embedder.encode(qdata['question']).clone().unsqueeze(0).to(device)

        # Forward
        selected, details = controller(
            query_emb, all_paths_by_k, embedder,
            deterministic=True, return_details=True)

        # Compute actual recall for final confidence target
        collected = set()
        for p in selected:
            for h, r, t in p['path_triples']:
                collected.add(str(h)); collected.add(str(t))
        actual_recall = len(collected & gold_entities) / max(len(gold_entities), 1)

        # Loss
        losses = loss_fn(details, gold_entities, all_paths_by_k)

        # Extra: final confidence calibration with actual recall
        conf_cal = F.mse_loss(
            details['final_confidence'],
            torch.tensor([actual_recall], device=device))
        total_loss = losses['total'] + 0.3 * conf_cal

        optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(controller.parameters(), 1.0)
        optimizer.step()

        ep_loss += total_loss.item()
        ep_conf += losses['confidence']
        ep_rel += losses['relevance']
        n_batch += 1

        if (qi + 1) % 300 == 0:
            print(f"    [{qi+1}/{len(train_qids)}] loss={ep_loss/n_batch:.4f}")

    scheduler.step()
    dt = time.time() - t0

    # Evaluate
    controller.eval()
    eval_ret = CGARRetriever(gnn_test, controller, embedder,
                              max_paths_per_pair=MAX_PATHS_PER_PAIR, device=device)
    eval_results = evaluate_cgar(eval_ret, gnn_test, test_qids[:200])

    m = ep_loss / max(n_batch, 1)
    print(f"  Epoch {epoch+1:2d}/{S1_EPOCHS} | Loss: {m:.4f} "
          f"(C:{ep_conf/max(n_batch,1):.3f} R:{ep_rel/max(n_batch,1):.3f}) | {dt:.1f}s")
    print(f"    Eval: F1={eval_results['f1']*100:.1f}% Recall={eval_results['recall']*100:.1f}% "
          f"Cost={eval_results['cost']:.1f} K={eval_results.get('chosen_k',0):.0f}")

    s1_history.append({
        'epoch': epoch+1, 'loss': m,
        'eval_f1': eval_results['f1'], 'eval_recall': eval_results['recall'],
        'eval_cost': eval_results['cost'],
    })

    if eval_results['f1'] > best_eval_f1:
        best_eval_f1 = eval_results['f1']
        torch.save({
            'epoch': epoch+1, 'model_state': controller.state_dict(),
            'eval_f1': eval_results['f1'],
        }, os.path.join(CHECKPOINT_DIR, 'cgar_s1_best.pt'))
        print(f"    >>> Best F1={eval_results['f1']*100:.1f}% saved")

embedder.save_cache(EMB_CACHE)
print(f"\n[STAGE 1 COMPLETE] Best F1: {best_eval_f1*100:.1f}%")

STAGE 1: SUPERVISED PRETRAINING
    [300/2848] loss=0.4536
    [600/2848] loss=0.4422
    [900/2848] loss=0.4435
    [1200/2848] loss=0.4385
    [1500/2848] loss=0.4349
    [1800/2848] loss=0.4343
    [2100/2848] loss=0.4338
    [2400/2848] loss=0.4330
    [2700/2848] loss=0.4318
  Epoch  1/10 | Loss: 0.4321 (C:0.121 R:0.675) | 241.0s
    Eval: F1=40.7% Recall=69.4% Cost=4.8 K=0
    >>> Best F1=40.7% saved
    [300/2848] loss=0.4139
    [600/2848] loss=0.4082
    [900/2848] loss=0.4146
    [1200/2848] loss=0.4123
    [1500/2848] loss=0.4124
    [1800/2848] loss=0.4141
    [2100/2848] loss=0.4149
    [2400/2848] loss=0.4133
    [2700/2848] loss=0.4136
  Epoch  2/10 | Loss: 0.4128 (C:0.099 R:0.674) | 215.7s
    Eval: F1=40.7% Recall=69.4% Cost=4.8 K=0
    [300/2848] loss=0.4010
    [600/2848] loss=0.4039
    [900/2848] loss=0.4071
    [1200/2848] loss=0.4054
    [1500/2848] loss=0.4031
    [1800/2848] loss=0.4013
    [2100/2848] loss=0.4011
    [2400/2848] loss=0.4002
    [2700/2848] los

---
## Stage 2: Self-Critical Sequence Training (SCST)
Fine-tune with SCST. Key advantage over P7: K varies stochastically among {5, 10, 20, 50}
so greedy and sampled produce DIFFERENT paths, giving a real advantage signal.

In [ ]:
print("=" * 60)
print("STAGE 2: SCST FINE-TUNING (with K variance)")
print("=" * 60)

s1_path = os.path.join(CHECKPOINT_DIR, 'cgar_s1_best.pt')
if os.path.exists(s1_path):
    ckpt = torch.load(s1_path, map_location='cpu', weights_only=False)
    controller.load_state_dict(ckpt['model_state'])
    print(f"  Loaded Stage 1 (F1={ckpt['eval_f1']*100:.1f}%)")

trainer = CGARSCSTTrainer(
    controller, gnn_train, embedder,
    alpha=ALPHA_REWARD, beta=BETA_REWARD, gamma=GAMMA_REWARD,
    n_samples=S2_SAMPLES, lr=S2_LR, device=str(device),
)

scst_history = []
best_scst = -float('inf')
random.shuffle(train_qids)

for step in range(1, S2_STEPS + 1):
    qid = train_qids[(step-1) % len(train_qids)]
    result = trainer.train_step(qid)
    if result is None: continue

    if step == 1 or step % 20 == 0:
        print(f"  SCST {step:4d}/{S2_STEPS} | R: {result['avg_reward']:.4f} | "
              f"Baseline: {result['baseline']:.4f} | Loss: {result['loss']:.4f}")
        scst_history.append(result)
        if result['avg_reward'] > best_scst:
            best_scst = result['avg_reward']
            torch.save({
                'step': step, 'model_state': controller.state_dict(),
                'reward': result['avg_reward'],
            }, os.path.join(CHECKPOINT_DIR, 'cgar_scst_best.pt'))

embedder.save_cache(EMB_CACHE)
print(f"\n[STAGE 2 COMPLETE] Best reward: {best_scst:.4f}")

STAGE 2: SCST FINE-TUNING (with K variance)
  Loaded Stage 1 (F1=41.0%)
  SCST    1/300 | R: 0.8800 | Baseline: 0.8800 | Loss: 0.0003
  SCST   20/300 | R: 0.2120 | Baseline: 0.2043 | Loss: 0.0042
  SCST   40/300 | R: 0.2800 | Baseline: 0.2800 | Loss: 0.0000
  SCST   60/300 | R: 0.8800 | Baseline: 0.8800 | Loss: 0.0000
  SCST   80/300 | R: 0.8200 | Baseline: 0.8200 | Loss: 0.0001
  SCST  100/300 | R: 0.7000 | Baseline: 0.7000 | Loss: 0.0001
  SCST  120/300 | R: 0.8600 | Baseline: 0.8600 | Loss: 0.0000
  SCST  140/300 | R: 0.8800 | Baseline: 0.8800 | Loss: 0.0000
  SCST  160/300 | R: 0.2800 | Baseline: 0.2800 | Loss: 0.0002
  SCST  180/300 | R: 0.3667 | Baseline: 0.3000 | Loss: 0.0015
  SCST  200/300 | R: 0.7000 | Baseline: 0.7000 | Loss: 0.0001
  SCST  220/300 | R: 0.2600 | Baseline: 0.2600 | Loss: 0.0001
  SCST  240/300 | R: 0.5600 | Baseline: 0.5600 | Loss: 0.0005
  SCST  260/300 | R: 0.8800 | Baseline: 0.8800 | Loss: 0.0000
  SCST  280/300 | R: 0.8800 | Baseline: 0.8800 | Loss: 0.000

---
## Evaluation: Retrieval Metrics
Compare CGAR with baselines on retrieval quality and cost.

In [ ]:
print("=" * 60)
print("EVALUATION: RETRIEVAL METRICS")
print("=" * 60)

# Load best model
for name, path in [('SCST', 'cgar_scst_best.pt'), ('S1', 'cgar_s1_best.pt')]:
    p = os.path.join(CHECKPOINT_DIR, path)
    if os.path.exists(p):
        ckpt = torch.load(p, map_location='cpu', weights_only=False)
        controller.load_state_dict(ckpt['model_state'])
        print(f"  Loaded {name} model"); break

controller.eval()

# Build retrievers
cgar_ret = CGARRetriever(gnn_test, controller, embedder,
                          max_paths_per_pair=MAX_PATHS_PER_PAIR, device=device)
greedy_ret = GreedyGNNRetriever(gnn_test, max_steps=5, budget=50)

class GNNRAGBaseline:
    def __init__(self, gnn, top_k=None, max_paths_per_pair=3):
        self.gnn = gnn; self.top_k = top_k; self.mpp = max_paths_per_pair
    def retrieve(self, qid):
        paths = extract_candidate_paths(qid, self.gnn, max_candidates=10, max_paths_per_pair=self.mpp)
        if self.top_k: paths.sort(key=lambda p: -p['gnn_score']); paths = paths[:self.top_k]
        triples = [t for p in paths for t in p['path_triples']]
        return {'triples': triples, 'cost': len(paths),
                'context': '\n'.join(p['path_text'] for p in paths)}

methods = {
    'GreedyGNN': greedy_ret,
    'GNN-RAG (all)': GNNRAGBaseline(gnn_test),
    'GNN-RAG (top-10)': GNNRAGBaseline(gnn_test, top_k=10),
    'GNN-RAG (top-5)': GNNRAGBaseline(gnn_test, top_k=5),
    'CGAR (\u03b2=0.85)': cgar_ret,
}

all_results = {}
for mname, ret in methods.items():
    print(f"\n  {mname}...")
    if 'CGAR' in mname:
        r = evaluate_cgar(ret, gnn_test, test_qids, verbose=True)
    else:
        from adagraph_controller import evaluate_retriever
        r = evaluate_retriever(ret, gnn_test, test_qids, verbose=True)
    all_results[mname] = r
    print(f"  \u2192 F1={r['f1']*100:.1f}% Recall={r['recall']*100:.1f}% Cost={r['cost']:.1f}")

print(f"\n{'='*70}")
print(f"  {'Method':<25} | {'F1':>6} | {'Recall':>6} | {'Prec':>6} | {'Cost':>6}")
print(f"  {'-'*65}")
for n, r in all_results.items():
    print(f"  {n:<25} | {r['f1']*100:5.1f}% | {r['recall']*100:5.1f}% | "
          f"{r['precision']*100:5.1f}% | {r['cost']:5.1f}")

with open(os.path.join(RESULTS_DIR, 'cgar_retrieval.json'), 'w') as f:
    json.dump({k: {mk: float(mv) for mk, mv in v.items()} for k, v in all_results.items()}, f, indent=2)

EVALUATION: RETRIEVAL METRICS
  Loaded SCST model

  GreedyGNN...


ModuleNotFoundError: No module named 'adagraph_controller'

---
## Pareto Analysis + Adaptive Behavior
Sweep confidence threshold beta to trace quality-cost Pareto frontier.
Analyze how CGAR adapts candidate budget K and path selection to query difficulty.

In [ ]:
print("=" * 60)
print("PARETO ANALYSIS + ADAPTIVE BEHAVIOR")
print("=" * 60)

import matplotlib
matplotlib.rcParams.update({'font.size': 12, 'figure.dpi': 150, 'font.family': 'serif'})
import matplotlib.pyplot as plt

# beta sweep — this is the KEY experiment
betas = [0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 0.99]
beta_results = {}
for beta in betas:
    controller.sprt_beta = beta
    ret = CGARRetriever(gnn_test, controller, embedder,
                         max_paths_per_pair=MAX_PATHS_PER_PAIR, device=device)
    r = evaluate_cgar(ret, gnn_test, test_qids[:300])
    beta_results[beta] = r
    print(f"  \u03b2={beta:.2f} \u2192 F1={r['f1']*100:.1f}% Cost={r['cost']:.1f} K={r.get('chosen_k',0):.0f}")
controller.sprt_beta = SPRT_BETA

# Per-query adaptive analysis
controller.eval()
query_analysis = []
with torch.no_grad():
    for qid in test_qids[:300]:
        result = cgar_ret.retrieve(qid, deterministic=True, return_details=True)
        gold = set(str(a) for a in gnn_test.get_answers(qid))
        if not gold: continue
        collected = set()
        for h, r, t in result['triples']:
            collected.add(str(h)); collected.add(str(t))
        query_analysis.append({
            'n_gold': len(gold),
            'chosen_k': result['details']['chosen_k'],
            'n_selected': result['n_selected'],
            'coverage': len(collected & gold) / max(len(gold), 1),
            'final_conf': result['details']['final_confidence'].item(),
        })

# Split by difficulty
n_golds = np.array([q['n_gold'] for q in query_analysis])
p33, p66 = np.percentile(n_golds, [33, 66])
easy = [q for q in query_analysis if q['n_gold'] <= p33]
medium = [q for q in query_analysis if p33 < q['n_gold'] <= p66]
hard = [q for q in query_analysis if q['n_gold'] > p66]

def avg(qs, k): return np.mean([q[k] for q in qs]) if qs else 0

print(f"\n  {'Metric':<25} | {'Easy':>8} | {'Med':>8} | {'Hard':>8} | {'All':>8}")
print(f"  {'-'*65}")
print(f"  {'Chosen K':<25} | {avg(easy,'chosen_k'):>7.0f} | {avg(medium,'chosen_k'):>7.0f} | "
      f"{avg(hard,'chosen_k'):>7.0f} | {avg(query_analysis,'chosen_k'):>7.0f}")
print(f"  {'Selected paths':<25} | {avg(easy,'n_selected'):>7.1f} | {avg(medium,'n_selected'):>7.1f} | "
      f"{avg(hard,'n_selected'):>7.1f} | {avg(query_analysis,'n_selected'):>7.1f}")
print(f"  {'Confidence':<25} | {avg(easy,'final_conf'):>7.3f} | {avg(medium,'final_conf'):>7.3f} | "
      f"{avg(hard,'final_conf'):>7.3f} | {avg(query_analysis,'final_conf'):>7.3f}")
print(f"  {'Coverage':<25} | {avg(easy,'coverage')*100:>6.1f}% | {avg(medium,'coverage')*100:>6.1f}% | "
      f"{avg(hard,'coverage')*100:>6.1f}% | {avg(query_analysis,'coverage')*100:>6.1f}%")

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('CGAR: Adaptive Behavior Analysis', fontsize=14, fontweight='bold')

# Pareto
ax = axes[0]
bc = [beta_results[b]['cost'] for b in betas]
bf = [beta_results[b]['f1']*100 for b in betas]
ax.plot(bc, bf, 's-', color='#2ecc71', lw=2, markersize=8, label='CGAR')
for b in betas:
    if b in [0.7, 0.85, 0.95]:
        ax.annotate(f'\u03b2={b}', (beta_results[b]['cost'], beta_results[b]['f1']*100),
                    textcoords="offset points", xytext=(5,-12), fontsize=9)
ax.set_xlabel('Cost (paths)'); ax.set_ylabel('F1 (%)')
ax.set_title('(a) Pareto Frontier'); ax.grid(True, alpha=0.3); ax.legend()

# K distribution by difficulty
ax = axes[1]
groups = {'Easy': easy, 'Medium': medium, 'Hard': hard}
x = np.arange(len(groups))
k_vals = [avg(g, 'chosen_k') for g in groups.values()]
ax.bar(x, k_vals, color=['#3498db', '#f39c12', '#e74c3c'], edgecolor='black')
ax.set_xticks(x); ax.set_xticklabels(groups.keys())
ax.set_ylabel('Avg Chosen K'); ax.set_title('(b) Candidate K by Difficulty')
ax.grid(True, alpha=0.3, axis='y')

# Selected paths by difficulty
ax = axes[2]
sel_vals = [avg(g, 'n_selected') for g in groups.values()]
ax.bar(x, sel_vals, color=['#3498db', '#f39c12', '#e74c3c'], edgecolor='black')
ax.set_xticks(x); ax.set_xticklabels(groups.keys())
ax.set_ylabel('Avg Selected Paths'); ax.set_title('(c) Budget Usage by Difficulty')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'cgar_analysis.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
print("=" * 60)
print("LLM END-TO-END EVALUATION")
print("=" * 60)

# ── LLM Setup ──
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

LLM_MODEL = 'google/flan-t5-large'
print(f"Loading {LLM_MODEL}...")

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
llm = AutoModelForSeq2SeqLM.from_pretrained(LLM_MODEL, torch_dtype=torch.float16, device_map='auto')
llm.eval()

def llm_generate(prompt, max_tokens=128):
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512)
    inputs = {k: v.to(llm.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = llm.generate(**inputs, max_new_tokens=max_tokens)
    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

# ── Prompt Template ──
INSTRUCTION = "Based on the reasoning paths, please answer the given question. Keep the answer as simple as possible."

def build_qa_prompt(question, paths_text):
    if not question.endswith('?'):
        question += '?'
    if paths_text:
        return f"{INSTRUCTION}\n\nReasoning Paths:\n{paths_text}\n\nQuestion:\n{question}"
    return f"{INSTRUCTION}\n\nQuestion:\n{question}"

# ── Evaluation Functions ──
def normalize(s):
    s = str(s).lower()
    s = ''.join(c for c in s if c not in set(string.punctuation))
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    return ' '.join(s.split())

def eval_hit(prediction, answers):
    pred_norm = normalize(prediction)
    return int(any(normalize(a) in pred_norm for a in answers))

def eval_f1_str(prediction, answers):
    pred_tokens = set(normalize(prediction).split())
    best_f1 = 0
    for a in answers:
        a_tokens = set(normalize(a).split())
        if not a_tokens:
            continue
        common = pred_tokens & a_tokens
        if not common:
            continue
        p = len(common) / len(pred_tokens) if pred_tokens else 0
        r = len(common) / len(a_tokens)
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
        best_f1 = max(best_f1, f1)
    return best_f1

# ── Build Retrievers for Comparison ──
gnnrag_all = GNNRAGBaseline(gnn_test)
gnnrag_top10 = GNNRAGBaseline(gnn_test, top_k=10)

eval_methods = {
    'GNN-RAG (all)': gnnrag_all,
    'GNN-RAG (top-10)': gnnrag_top10,
    'CGAR': cgar_ret,
}

n_eval = min(300, len(test_qids))
llm_results = {}

for method_name, ret in eval_methods.items():
    print(f"\n  Evaluating {method_name}...")
    hits, f1s, costs = [], [], []

    for i, qid in enumerate(test_qids[:n_eval]):
        qdata = gnn_test.get(qid)
        if qdata is None:
            continue

        # Retrieve
        if 'CGAR' in method_name:
            result = ret.retrieve(qid, deterministic=True)
        else:
            result = ret.retrieve(qid)
        answers = [str(a) for a in gnn_test.get_answers(qid)]
        if not answers:
            continue

        # Build prompt
        prompt = build_qa_prompt(qdata['question'], result.get('context', ''))

        # Generate
        prediction = llm_generate(prompt)

        # Evaluate
        hits.append(eval_hit(prediction, answers))
        f1s.append(eval_f1_str(prediction, answers))
        costs.append(result['cost'])

        if i < 3:
            print(f"    Q: {qdata['question'][:80]}...")
            print(f"    A: {answers[:3]}, Pred: {prediction[:80]}")
            print(f"    Hit={hits[-1]}, F1={f1s[-1]:.3f}, Paths={result['cost']}")

    llm_results[method_name] = {
        'hit': np.mean(hits) * 100 if hits else 0,
        'f1': np.mean(f1s) * 100 if f1s else 0,
        'cost': np.mean(costs) if costs else 0,
        'n': len(hits),
    }

# Results table
print(f"\n{'='*70}")
print(f"  {'Method':<25} | {'Hits@1':>7} | {'F1':>7} | {'Avg Paths':>10}")
print(f"  {'-'*60}")
for name, r in llm_results.items():
    print(f"  {name:<25} | {r['hit']:>6.1f}% | {r['f1']:>6.1f}% | {r['cost']:>10.1f}")

with open(os.path.join(RESULTS_DIR, 'cgar_llm_eval.json'), 'w') as f:
    json.dump(llm_results, f, indent=2)
print(f"\nLLM results saved")

In [ ]:
print("=" * 60)
print("ABLATION STUDY")
print("=" * 60)

# Ablation 1: No Level 1 (fixed K=10, skip OptStop)
print("\n  Ablation: Fixed K=10 (no OptStop)...")
controller.sprt_beta = 0.0  # never stop -> always use max K
r_no_optstop = evaluate_cgar(
    CGARRetriever(gnn_test, controller, embedder, max_paths_per_pair=MAX_PATHS_PER_PAIR, device=device),
    gnn_test, test_qids[:300])
controller.sprt_beta = SPRT_BETA
print(f"    F1={r_no_optstop['f1']*100:.1f}% Cost={r_no_optstop['cost']:.1f}")

# Ablation 2: No Level 2 (no relevance ranking, just top by GNN score)
print("\n  Ablation: No relevance ranking (GNN score only)...")
r_no_ranking = evaluate_cgar(
    CGARRetriever(gnn_test, controller, embedder, max_paths_per_pair=MAX_PATHS_PER_PAIR, device=device),
    gnn_test, test_qids[:300])
print(f"    F1={r_no_ranking['f1']*100:.1f}% Cost={r_no_ranking['cost']:.1f}")

# Ablation 3: No redundancy penalty (lambda=0)
print("\n  Ablation: No redundancy penalty (\u03bb=0)...")
orig_lambda = controller.redundancy_lambda
controller.redundancy_lambda = 0.0
r_no_redund = evaluate_cgar(
    CGARRetriever(gnn_test, controller, embedder, max_paths_per_pair=MAX_PATHS_PER_PAIR, device=device),
    gnn_test, test_qids[:300])
controller.redundancy_lambda = orig_lambda
print(f"    F1={r_no_redund['f1']*100:.1f}% Cost={r_no_redund['cost']:.1f}")

# Summary
print(f"\n  {'Configuration':<35} | {'F1':>6} | {'Cost':>6}")
print(f"  {'-'*55}")
print(f"  {'CGAR (full)':<35} | {all_results.get('CGAR (\u03b2=0.85)',{}).get('f1',0)*100:5.1f}% | "
      f"{all_results.get('CGAR (\u03b2=0.85)',{}).get('cost',0):5.1f}")
print(f"  {'- No OptStop (fixed K=max)':<35} | {r_no_optstop['f1']*100:5.1f}% | {r_no_optstop['cost']:5.1f}")
print(f"  {'- No redundancy (\u03bb=0)':<35} | {r_no_redund['f1']*100:5.1f}% | {r_no_redund['cost']:5.1f}")

In [ ]:
print("=" * 60)
print("PAPER FIGURES")
print("=" * 60)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('CGAR: Confidence-Guided Adaptive Retrieval \u2014 Paper Results', fontsize=16, fontweight='bold')

# Fig 1: Training loss curve
ax = axes[0, 0]
if s1_history:
    eps = [h['epoch'] for h in s1_history]
    ax.plot(eps, [h['loss'] for h in s1_history], 'b-o', lw=2)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.set_title('(a) Stage 1 Training Loss')
    ax.grid(True, alpha=0.3)

# Fig 2: Training eval metrics
ax = axes[0, 1]
if s1_history:
    ax.plot(eps, [h['eval_f1']*100 for h in s1_history], 'g-s', lw=2, label='F1')
    ax.plot(eps, [h['eval_recall']*100 for h in s1_history], 'r-^', lw=2, label='Recall')
    ax2 = ax.twinx()
    ax2.plot(eps, [h['eval_cost'] for h in s1_history], 'b--', lw=1, alpha=0.5, label='Cost')
    ax2.set_ylabel('Avg Cost', color='blue')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Score (%)')
    ax.set_title('(b) Evaluation During Training')
    ax.legend(loc='lower right'); ax.grid(True, alpha=0.3)

# Fig 3: SCST reward curve
ax = axes[0, 2]
if scst_history:
    steps = list(range(len(scst_history)))
    ax.plot(steps, [h['avg_reward'] for h in scst_history], 'g-', lw=2, label='Sample R')
    ax.plot(steps, [h['baseline'] for h in scst_history], 'r--', lw=1, label='Baseline R')
    ax.set_xlabel('SCST Step (\u00d720)'); ax.set_ylabel('Reward')
    ax.set_title('(c) SCST Fine-tuning')
    ax.legend(); ax.grid(True, alpha=0.3)

# Fig 4: Method comparison bars
ax = axes[1, 0]
if all_results:
    names = list(all_results.keys())
    f1_vals = [all_results[n]['f1']*100 for n in names]
    colors = ['#95a5a6'] * (len(names)-1) + ['#2ecc71']
    ax.bar(range(len(names)), f1_vals, color=colors, edgecolor='black', linewidth=0.5)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=25, ha='right', fontsize=9)
    ax.set_ylabel('F1 (%)')
    ax.set_title('(d) Retrieval F1 Comparison')
    ax.grid(True, alpha=0.3, axis='y')

# Fig 5: Pareto frontier
ax = axes[1, 1]
if beta_results:
    bc = [beta_results[b]['cost'] for b in betas]
    bf = [beta_results[b]['f1']*100 for b in betas]
    ax.plot(bc, bf, 's-', color='#2ecc71', lw=2, markersize=8, label='CGAR (\u03b2 sweep)')
    for b in betas:
        if b in [0.7, 0.85, 0.95]:
            ax.annotate(f'\u03b2={b}', (beta_results[b]['cost'], beta_results[b]['f1']*100),
                        textcoords="offset points", xytext=(5,-12), fontsize=9)
    ax.set_xlabel('Cost (paths)'); ax.set_ylabel('F1 (%)')
    ax.set_title('(e) Pareto Frontier')
    ax.legend(); ax.grid(True, alpha=0.3)

# Fig 6: LLM end-to-end
ax = axes[1, 2]
if llm_results:
    names = list(llm_results.keys())
    hit_vals = [llm_results[n]['hit'] for n in names]
    f1_vals = [llm_results[n]['f1'] for n in names]
    x = np.arange(len(names))
    ax.bar(x - 0.15, hit_vals, 0.3, label='Hits@1', color='#3498db')
    ax.bar(x + 0.15, f1_vals, 0.3, label='F1', color='#e74c3c')
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=25, ha='right', fontsize=9)
    ax.set_ylabel('Score (%)')
    ax.set_title('(f) LLM End-to-End (Flan-T5)')
    ax.legend(); ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'cgar_paper_figures.png'), dpi=300, bbox_inches='tight')
plt.show()
print("[OK] Paper figures saved")

In [ ]:
print("=" * 60)
print("SAVING ALL RESULTS")
print("=" * 60)

final_results = {
    'config': {
        'emb_dim': EMB_DIM, 'hidden_dim': HIDDEN_DIM,
        'max_paths': MAX_PATHS, 'sprt_beta': SPRT_BETA,
        'redundancy_lambda': REDUNDANCY_LAMBDA,
        's1_epochs': S1_EPOCHS, 's2_steps': S2_STEPS,
        'k_options': CGARController.K_OPTIONS,
    },
    's1_history': s1_history,
    'scst_history': scst_history,
    'retrieval_results': {k: {mk: float(mv) for mk, mv in v.items()}
                          for k, v in all_results.items()} if all_results else {},
    'llm_results': llm_results if llm_results else {},
}

with open(os.path.join(RESULTS_DIR, 'cgar_full_results.json'), 'w') as f:
    json.dump(final_results, f, indent=2)

# Save final checkpoint
torch.save({
    'model_state': controller.state_dict(),
    'config': final_results['config'],
}, os.path.join(CHECKPOINT_DIR, 'cgar_final.pt'))

embedder.save_cache(EMB_CACHE)

print(f"  Results: {RESULTS_DIR}/cgar_full_results.json")
print(f"  Model:   {CHECKPOINT_DIR}/cgar_final.pt")
print(f"  Cache:   {EMB_CACHE}")
print(f"\n[ALL DONE]")